# Benchmark offline (Kaggle, bez Hugging Face)

Pliki: `/kaggle/input/datasets/kamilml/sources/`
Wyniki: `/kaggle/working/Steganography_colab/`

In [ ]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources')
PROJECT_DIR = Path('/kaggle/working/Steganography_colab')
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
REQUIRED = ['run_experiments.py', 'dummy_processor.py', 'requirements.txt']

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in REQUIRED if not (INPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje w {INPUT_DIR}: {missing}')

for name in REQUIRED:
    shutil.copy2(INPUT_DIR / name, PROJECT_DIR / name)
    print(f'copied {name}')

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

print('cwd:', Path.cwd())
print('input:', INPUT_DIR)

In [ ]:
!pip install -q -r requirements.txt

## Konfiguracja przebiegu

In [ ]:
TEST = 'humaneval'
MODEL = 'gemma'
THRESHOLD = 0.0
TOP_N = 16
MAX_NEW_TOKENS = 512
HUMANEVAL_TASKS = None

In [ ]:
!python run_experiments.py \
  --offline \
  --list-cached-models \
  --model-cache-dir {MODEL_CACHE_DIR}

In [ ]:
_humaneval_args = f' --humaneval-tasks {HUMANEVAL_TASKS}' if HUMANEVAL_TASKS else ''

!python run_experiments.py \
  --offline \
  --test {TEST} \
  --model {MODEL} \
  --threshold {THRESHOLD} \
  --top-n {TOP_N} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --model-cache-dir {MODEL_CACHE_DIR}{_humaneval_args}

In [ ]:
import csv

summary_csv = Path('results/summary.csv')
if summary_csv.exists():
    with summary_csv.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Rows in summary.csv: {len(rows)}')
    for row in rows[-5:]:
        print(row)
else:
    print('summary.csv not found yet')